# 🎣 Sistema de Predicción de Pesca desde Orilla
## Costa Sur de Perú (Tacna - Ilo)

Este notebook permite:
- Descargar y actualizar datos oceanográficos (SST, Clorofila)
- Visualizar la línea costera real
- Analizar spots de pesca con transectos
- Predecir movimiento de peces
- Generar mapas interactivos

## 1. Configuración Inicial

In [ ]:
# Instalar dependencias si es necesario
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Paquetes necesarios
for pkg in ['folium', 'pandas', 'numpy', 'requests', 'pyarrow']:
    install_if_missing(pkg)

print("✅ Dependencias listas")

In [ ]:
# Imports
import os
import sys
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import folium
from folium import plugins
import json

# Agregar path del proyecto
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# Imports del proyecto
from data.data_manager import DataManager
from core.coastline_real import RealCoastline, load_coastline

print(f"📅 Fecha actual: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("✅ Módulos cargados")

## 2. Gestión de Datos

### 2.1 Estado Actual de los Datos

In [ ]:
# Inicializar gestor de datos
dm = DataManager()

# Ver estado actual
summary = dm.get_data_summary()

print("📊 ESTADO DE LOS DATOS")
print("=" * 50)
print(f"Directorio cache: {summary['cache_dir']}")
print(f"Última actualización: {summary['last_update'] or 'Nunca'}")
print(f"Coastline: {summary['coastline_version'] or 'No descargado'}")
print()

for ds_name, ds_info in summary['datasets'].items():
    print(f"📈 {ds_name.upper()}:")
    if 'status' in ds_info:
        print(f"   Estado: {ds_info['status']}")
    else:
        print(f"   Registros: {ds_info['records']:,}")
        print(f"   Rango: {ds_info['date_min'][:10]} a {ds_info['date_max'][:10]}")
        print(f"   Tamaño: {ds_info['file_size_mb']:.2f} MB")
    print()

### 2.2 Descargar/Actualizar Datos

Ejecuta esta celda para descargar datos nuevos o actualizar los existentes.

In [ ]:
# Configuración de descarga
MESES_ATRAS = 4  # Cuántos meses de datos descargar
FORZAR_DESCARGA_COMPLETA = False  # True para re-descargar todo

print("🌊 DESCARGANDO DATOS OCEANOGRÁFICOS")
print("=" * 50)
print(f"Periodo: últimos {MESES_ATRAS} meses")
print(f"Modo: {'Completo' if FORZAR_DESCARGA_COMPLETA else 'Incremental'}")
print()

# Descargar/actualizar
data = dm.update_oceanographic_data(
    months_back=MESES_ATRAS,
    force_full=FORZAR_DESCARGA_COMPLETA
)

print()
print("✅ Descarga completada")

# Mostrar resumen
for name, df in data.items():
    if df is not None:
        print(f"   {name}: {len(df):,} registros")

### 2.3 Descargar Línea Costera

In [ ]:
print("🏖️ DESCARGANDO LÍNEA COSTERA")
print("=" * 50)

# Descargar coastline de OpenStreetMap
coastline_path = dm.download_coastline(force=False)

# Cargar modelo de costa
coastline = RealCoastline(str(coastline_path))

print(f"\n✅ Línea costera cargada")
print(f"   Puntos de costa: {len(coastline.points)}")

if coastline.points:
    lats = [p.lat for p in coastline.points]
    lons = [p.lon for p in coastline.points]
    print(f"   Rango latitud: {min(lats):.4f} a {max(lats):.4f}")
    print(f"   Rango longitud: {min(lons):.4f} a {max(lons):.4f}")

## 3. Visualización de la Costa

### 3.1 Mapa de Línea Costera

In [ ]:
# Crear mapa base
center_lat = -17.85
center_lon = -71.10

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles=None
)

# Capa satelite
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satélite"
).add_to(m)

# Capa calles
folium.TileLayer(tiles="OpenStreetMap", name="Calles").add_to(m)

# Dibujar línea costera
if coastline.points:
    coast_coords = [[p.lat, p.lon] for p in coastline.points]
    folium.PolyLine(
        locations=coast_coords,
        color="yellow",
        weight=3,
        opacity=0.8,
        popup="Línea de Costa"
    ).add_to(m)

# Agregar algunos puntos de muestra
sample_points = coastline.sample_coast(num_points=15)
for i, point in enumerate(sample_points):
    folium.CircleMarker(
        location=[point.lat, point.lon],
        radius=6,
        color="red",
        fill=True,
        fillColor="yellow",
        fillOpacity=0.8,
        popup=f"Punto {i+1}<br>Lat: {point.lat:.5f}<br>Lon: {point.lon:.5f}<br>Bearing mar: {point.bearing_to_sea:.0f}°"
    ).add_to(m)

folium.LayerControl().add_to(m)

print("🗺️ Mapa de línea costera:")
m

### 3.2 Mapa con Transectos

In [ ]:
# Crear mapa con transectos
m2 = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# Muestrear puntos para análisis
analysis_points = coastline.sample_coast(num_points=20)

# Crear transectos y dibujar
for i, point in enumerate(analysis_points):
    # Crear transecto perpendicular
    transect = coastline.create_transect_from_point(
        point,
        distance_m=500,
        num_points=5
    )
    
    # Dibujar línea del transecto
    transect_coords = [[t[0], t[1]] for t in transect]
    folium.PolyLine(
        locations=transect_coords,
        color="cyan",
        weight=2,
        opacity=0.7,
        popup=f"Transecto {i+1}"
    ).add_to(m2)
    
    # Marcador en la orilla
    folium.CircleMarker(
        location=[point.lat, point.lon],
        radius=8,
        color="#000",
        weight=2,
        fill=True,
        fillColor="#00FF00",
        fillOpacity=0.9,
        popup=f"<b>Spot {i+1}</b><br>Lat: {point.lat:.5f}<br>Lon: {point.lon:.5f}"
    ).add_to(m2)
    
    # Flecha de dirección hacia el mar
    arrow_end = transect[-1]
    folium.PolyLine(
        locations=[[point.lat, point.lon], [arrow_end[0], arrow_end[1]]],
        color="#FF6600",
        weight=3,
        opacity=0.8
    ).add_to(m2)

print("🗺️ Mapa con transectos perpendiculares:")
print(f"   {len(analysis_points)} spots de pesca")
print(f"   Transectos de 500m hacia el mar")
m2

## 4. Análisis de Datos Oceanográficos

### 4.1 Ver Datos SST más Recientes

In [ ]:
# Obtener datos SST recientes
sst_df = dm.get_latest_data('sst', days_back=8)

if sst_df is not None:
    print("🌡️ DATOS SST (últimos 8 días)")
    print("=" * 50)
    print(f"Registros: {len(sst_df):,}")
    print(f"SST promedio: {sst_df['sst'].mean():.2f}°C")
    print(f"SST mínima: {sst_df['sst'].min():.2f}°C")
    print(f"SST máxima: {sst_df['sst'].max():.2f}°C")
    print()
    print(sst_df.describe())
else:
    print("⚠️ No hay datos SST disponibles. Ejecuta la celda 2.2 para descargar.")

In [ ]:
# Obtener datos Clorofila recientes
chl_df = dm.get_latest_data('chlorophyll', days_back=8)

if chl_df is not None:
    print("🌿 DATOS CLOROFILA (últimos 8 días)")
    print("=" * 50)
    print(f"Registros: {len(chl_df):,}")
    print(f"Clorofila promedio: {chl_df['chlorophyll'].mean():.2f} mg/m³")
    print(f"Clorofila mínima: {chl_df['chlorophyll'].min():.2f} mg/m³")
    print(f"Clorofila máxima: {chl_df['chlorophyll'].max():.2f} mg/m³")
    print()
    print(chl_df.describe())
else:
    print("⚠️ No hay datos de clorofila disponibles. Ejecuta la celda 2.2 para descargar.")

### 4.2 Mapa de Calor SST

In [ ]:
if sst_df is not None and len(sst_df) > 0:
    # Crear mapa de calor
    m3 = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles="CartoDB dark_matter"
    )
    
    # Preparar datos para heatmap
    heat_data = []
    for _, row in sst_df.iterrows():
        # Normalizar SST para intensidad (14-19°C -> 0-1)
        intensity = (row['sst'] - 14) / 5
        intensity = max(0, min(1, intensity))
        heat_data.append([row['lat'], row['lon'], intensity])
    
    # Agregar heatmap
    plugins.HeatMap(
        heat_data,
        min_opacity=0.4,
        radius=15,
        blur=10,
        gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
    ).add_to(m3)
    
    # Agregar línea de costa
    if coastline.points:
        coast_coords = [[p.lat, p.lon] for p in coastline.points]
        folium.PolyLine(
            locations=coast_coords,
            color="white",
            weight=2,
            opacity=0.7
        ).add_to(m3)
    
    print("🌡️ Mapa de calor SST:")
    print("   Azul = Agua fría (14°C)")
    print("   Rojo = Agua cálida (19°C)")
    m3
else:
    print("⚠️ Descarga datos SST primero (celda 2.2)")

## 5. Análisis de Spot Específico

Ingresa las coordenadas de un punto para analizar.

In [ ]:
# === CONFIGURACIÓN DEL ANÁLISIS ===
# Modifica estas coordenadas para analizar otro punto

PUNTO_ANALISIS = {
    "lat": -18.21437,
    "lon": -70.57990,
    "nombre": "Mi Punto de Pesca"
}

print(f"📍 Punto a analizar: {PUNTO_ANALISIS['nombre']}")
print(f"   Coordenadas: {PUNTO_ANALISIS['lat']}, {PUNTO_ANALISIS['lon']}")

In [ ]:
# Encontrar punto más cercano EN la costa
punto_costa = coastline.get_point_on_coast(
    PUNTO_ANALISIS['lat'],
    PUNTO_ANALISIS['lon']
)

print("🏖️ PUNTO EN LA COSTA MÁS CERCANO")
print("=" * 50)
print(f"Latitud: {punto_costa.lat:.6f}")
print(f"Longitud: {punto_costa.lon:.6f}")
print(f"Bearing hacia el mar: {punto_costa.bearing_to_sea:.1f}°")
print(f"Bearing a lo largo de la costa: {punto_costa.bearing_along_coast:.1f}°")

# Crear transecto
transecto = coastline.create_transect_from_point(
    punto_costa,
    distance_m=500,
    num_points=10
)

print(f"\n📏 Transecto generado: {len(transecto)} puntos, 500m hacia el mar")

In [ ]:
# Mapa del punto específico
m4 = folium.Map(
    location=[punto_costa.lat, punto_costa.lon],
    zoom_start=16,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# Punto original (referencia)
folium.Marker(
    location=[PUNTO_ANALISIS['lat'], PUNTO_ANALISIS['lon']],
    popup=f"Punto original: {PUNTO_ANALISIS['nombre']}",
    icon=folium.Icon(color='blue', icon='info-sign')
).add_to(m4)

# Punto en la costa (corregido)
folium.Marker(
    location=[punto_costa.lat, punto_costa.lon],
    popup=f"<b>Punto en costa</b><br>Lat: {punto_costa.lat:.5f}<br>Lon: {punto_costa.lon:.5f}",
    icon=folium.Icon(color='green', icon='anchor')
).add_to(m4)

# Transecto
transect_coords = [[t[0], t[1]] for t in transecto]
folium.PolyLine(
    locations=transect_coords,
    color="cyan",
    weight=4,
    opacity=0.8,
    popup="Transecto hacia el mar"
).add_to(m4)

# Puntos del transecto
for i, (lat, lon, dist) in enumerate(transecto):
    color = "green" if i == 0 else "orange"
    folium.CircleMarker(
        location=[lat, lon],
        radius=6 if i == 0 else 4,
        color=color,
        fill=True,
        fillColor=color,
        popup=f"Punto {i+1}<br>Distancia: {dist:.0f}m"
    ).add_to(m4)

print(f"🗺️ Mapa del análisis de {PUNTO_ANALISIS['nombre']}:")
print("   🔵 Azul = Punto original")
print("   🟢 Verde = Punto corregido en la costa")
print("   🟠 Naranja = Puntos del transecto")
m4

## 6. Guardar Mapa

Exporta el mapa actual a un archivo HTML.

In [ ]:
# Guardar el último mapa generado
output_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'output')
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, 'analisis_jupyter.html')
m4.save(output_file)

print(f"✅ Mapa guardado en: {output_file}")

## 7. Comandos Rápidos

Funciones de utilidad para análisis rápidos.

In [ ]:
def analizar_punto_rapido(lat, lon, nombre="Punto"):
    """Análisis rápido de un punto."""
    punto = coastline.get_point_on_coast(lat, lon)
    
    print(f"📍 {nombre}")
    print(f"   Original: {lat:.5f}, {lon:.5f}")
    print(f"   En costa: {punto.lat:.5f}, {punto.lon:.5f}")
    print(f"   Bearing mar: {punto.bearing_to_sea:.0f}°")
    
    return punto

def actualizar_datos():
    """Actualiza todos los datos."""
    print("🔄 Actualizando datos...")
    dm.download_coastline()
    dm.update_oceanographic_data(months_back=4)
    print("✅ Datos actualizados")

print("📋 Funciones disponibles:")
print("   analizar_punto_rapido(lat, lon, nombre)")
print("   actualizar_datos()")

In [ ]:
# Ejemplo de uso
analizar_punto_rapido(-17.85, -71.15, "Ilo Centro")